# Task 2 Notebook for Python Web Scraping Assignment
## Caolán Maguire 25256569

In [3]:
# Libraries
import os
from os import listdir
from os.path import isfile, isdir, join
import json

In [7]:
# Load the saved dataset from Task 1
mypath = 'output'
onlydirs = [d for d in listdir(mypath) if isdir(join(mypath, d))]
print(onlydirs)

# Dictionary to store all listings by quarter
quarterly_data = {
    'Q1': [],
    'Q2': [],
    'Q3': [],
    'Q4': []
}

# Load all JSON files
for dirs in onlydirs:
    print(' +' + dirs)
    mypath = 'output/'+ dirs
    onlyfiles = [f for f in os.listdir(mypath) if os.path.isfile(os.path.join(mypath, f))]
    
    for file in onlyfiles:
        # Only process JSON files
        if file.endswith('.json'):
            filepath = os.path.join(mypath, file)
            with open(filepath, 'r') as f:
                listing = json.load(f)
                quarterly_data[dirs].append(listing)
    
    print(' - Loaded ' + str(len(quarterly_data[dirs])) + ' listings from ' + dirs)

# Function to extract price as a number
def extract_price(price_str):
    # Remove €, \u20ac (Euro symbol), commas, 'per month', 'EUR', and extra spaces
    cleaned = price_str.replace('€', '').replace('\u20ac', '').replace('EUR', '').replace(',', '').replace('per month', '').strip()
    try:
        return float(cleaned)
    except:
        return None

# Function to extract bedrooms as a number
def extract_bedrooms(bed_str):
    # Extract number from strings like "3 Bedrooms", "2", "1 Bedroom"
    import re
    match = re.search(r'\d+', bed_str)
    if match:
        return int(match.group())
    return None

# Function to extract month from record string
def extract_month(record_str):
    # Extract month from "January 2025 — Apartment" format
    import re
    months = ['January', 'February', 'March', 'April', 'May', 'June', 
              'July', 'August', 'September', 'October', 'November', 'December']
    for month in months:
        if month in record_str:
            return month
    return None

# Calculate average rent per quarter
print('\n' + '='*50)
print('AVERAGE RENT BY QUARTER')
print('='*50)

quarter_averages = {}

for quarter in ['Q1', 'Q2', 'Q3', 'Q4']:
    prices = []
    for listing in quarterly_data[quarter]:
        price = extract_price(listing.get('Price', '0'))
        if price:
            prices.append(price)
    
    if len(prices) > 0:
        avg_price = sum(prices) / len(prices)
        quarter_averages[quarter] = avg_price
        print(quarter + ': €' + str(round(avg_price, 2)) + ' (from ' + str(len(prices)) + ' listings)')
    else:
        quarter_averages[quarter] = 0
        print(quarter + ': No data')

# Calculate average rent per month
print('\n' + '='*50)
print('AVERAGE RENT BY MONTH')
print('='*50)

monthly_data = {}

for quarter in ['Q1', 'Q2', 'Q3', 'Q4']:
    for listing in quarterly_data[quarter]:
        month = extract_month(listing.get('record', ''))
        price = extract_price(listing.get('Price', '0'))
        
        if month and price:
            if month not in monthly_data:
                monthly_data[month] = []
            monthly_data[month].append(price)

# Calculate averages and display
month_order = ['January', 'February', 'March', 'April', 'May', 'June', 
               'July', 'August', 'September', 'October', 'November', 'December']

monthly_averages = {}

for month in month_order:
    if month in monthly_data:
        avg_price = sum(monthly_data[month]) / len(monthly_data[month])
        monthly_averages[month] = avg_price
        print(month + ': €' + str(round(avg_price, 2)) + ' (from ' + str(len(monthly_data[month])) + ' listings)')
    else:
        print(month + ': No data')

# Calculate average rent per quarter PER BEDROOM COUNT
print('\n' + '='*50)
print('AVERAGE RENT BY QUARTER AND BEDROOMS')
print('='*50)

# Structure: bedroom_quarterly_data[bedrooms][quarter] = [prices]
bedroom_quarterly_data = {}

for quarter in ['Q1', 'Q2', 'Q3', 'Q4']:
    for listing in quarterly_data[quarter]:
        price = extract_price(listing.get('Price', '0'))
        bedrooms = extract_bedrooms(listing.get('Bedrooms', '0'))
        
        if price and bedrooms:
            if bedrooms not in bedroom_quarterly_data:
                bedroom_quarterly_data[bedrooms] = {'Q1': [], 'Q2': [], 'Q3': [], 'Q4': []}
            bedroom_quarterly_data[bedrooms][quarter].append(price)

# Calculate averages
bedroom_quarterly_averages = {}

for bedrooms in sorted(bedroom_quarterly_data.keys()):
    print('\n' + str(bedrooms) + ' Bedroom(s):')
    bedroom_quarterly_averages[bedrooms] = {}
    
    for quarter in ['Q1', 'Q2', 'Q3', 'Q4']:
        prices = bedroom_quarterly_data[bedrooms][quarter]
        if len(prices) > 0:
            avg_price = sum(prices) / len(prices)
            bedroom_quarterly_averages[bedrooms][quarter] = avg_price
            print('  ' + quarter + ': €' + str(round(avg_price, 2)) + ' (from ' + str(len(prices)) + ' listings)')
        else:
            bedroom_quarterly_averages[bedrooms][quarter] = None
            print('  ' + quarter + ': No data')

# Visualize with matplotlib
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    
    # Create figure with 3 subplots
    fig = plt.figure(figsize=(18, 6))
    
    # Plot 1: Quarterly averages
    ax1 = plt.subplot(1, 3, 1)
    quarters = list(quarter_averages.keys())
    avg_prices_q = list(quarter_averages.values())
    colors_q = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
    
    ax1.bar(quarters, avg_prices_q, color=colors_q, edgecolor='black', linewidth=1.5)
    ax1.set_xlabel('Quarter', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Average Rent (€)', fontsize=12, fontweight='bold')
    ax1.set_title('Average Rental Price by Quarter - 2025', fontweight='bold', fontsize=14)
    ax1.grid(True, alpha=0.3, axis='y')
    
    for i, v in enumerate(avg_prices_q):
        ax1.text(i, v + 50, '€' + str(int(round(v, 0))), ha='center', fontweight='bold')
    
    # Plot 2: Monthly averages
    ax2 = plt.subplot(1, 3, 2)
    months_with_data = [m for m in month_order if m in monthly_averages]
    avg_prices_m = [monthly_averages[m] for m in months_with_data]
    
    ax2.bar(range(len(months_with_data)), avg_prices_m, color='#9B59B6', edgecolor='black', linewidth=1.5)
    ax2.set_xlabel('Month', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Average Rent (€)', fontsize=12, fontweight='bold')
    ax2.set_title('Average Rental Price by Month - 2025', fontweight='bold', fontsize=14)
    ax2.set_xticks(range(len(months_with_data)))
    ax2.set_xticklabels([m[:3] for m in months_with_data], rotation=45)
    ax2.grid(True, alpha=0.3, axis='y')
    
    for i, v in enumerate(avg_prices_m):
        ax2.text(i, v + 50, '€' + str(int(round(v, 0))), ha='center', fontweight='bold', fontsize=9)
    
    # Plot 3: Line graph by bedrooms across quarters
    ax3 = plt.subplot(1, 3, 3)
    
    # Line colors for different bedroom counts
    line_colors = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12', '#9B59B6', '#1ABC9C']
    marker_styles = ['o', 's', '^', 'D', 'v', 'p']
    
    quarters_list = ['Q1', 'Q2', 'Q3', 'Q4']
    
    for idx, bedrooms in enumerate(sorted(bedroom_quarterly_averages.keys())):
        prices_by_quarter = []
        quarters_with_data = []
        
        for quarter in quarters_list:
            avg = bedroom_quarterly_averages[bedrooms].get(quarter)
            if avg is not None:
                prices_by_quarter.append(avg)
                quarters_with_data.append(quarter)
        
        if len(prices_by_quarter) > 0:
            color = line_colors[idx % len(line_colors)]
            marker = marker_styles[idx % len(marker_styles)]
            
            # Plot line
            ax3.plot(quarters_with_data, prices_by_quarter, 
                    marker=marker, linewidth=2.5, markersize=8,
                    color=color, label=str(bedrooms) + ' Bed(s)',
                    markeredgecolor='black', markeredgewidth=1)
    
    ax3.set_xlabel('Quarter', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Average Rent (€)', fontsize=12, fontweight='bold')
    ax3.set_title('Average Rent by Bedrooms Across Quarters', fontweight='bold', fontsize=14)
    ax3.legend(loc='best', fontsize=10, frameon=True, shadow=True)
    ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('output/rental_analysis_complete.png', dpi=300, bbox_inches='tight')
    print('\nGraphs saved to output/rental_analysis_complete.png')
    
except ImportError as e:
    print('\nMatplotlib import failed: ' + str(e))
    print('Please run: pip install "numpy<2" and restart your kernel')

['Q1', 'Q2', 'Q3', 'Q4']
 +Q1
 - Loaded 1008 listings from Q1
 +Q2
 - Loaded 990 listings from Q2
 +Q3
 - Loaded 910 listings from Q3
 +Q4
 - Loaded 886 listings from Q4

AVERAGE RENT BY QUARTER
Q1: €2328.79 (from 1008 listings)
Q2: €2480.73 (from 990 listings)
Q3: €2489.14 (from 910 listings)
Q4: €2581.96 (from 886 listings)

AVERAGE RENT BY MONTH
January: €2366.72 (from 360 listings)
February: €2336.81 (from 320 listings)
March: €2279.33 (from 328 listings)
April: €2474.13 (from 358 listings)
May: €2445.85 (from 342 listings)
June: €2530.0 (from 290 listings)
July: €2566.82 (from 314 listings)
August: €2288.41 (from 276 listings)
September: €2586.06 (from 320 listings)
October: €2554.16 (from 308 listings)
November: €2597.61 (from 360 listings)
December: €2595.41 (from 218 listings)

AVERAGE RENT BY QUARTER AND BEDROOMS

1 Bedroom(s):
  Q1: €1493.91 (from 394 listings)
  Q2: €1621.26 (from 382 listings)
  Q3: €1642.11 (from 332 listings)
  Q4: €1686.19 (from 354 listings)

2 Bedroom(

In [ ]:
!pip uninstall numpy
!pip install "numpy<2.0"

In [ ]:
# Analyse

# Characterise

# Summarise

# Visualisatins where appropriate

In [ ]:
# References

# https://stackoverflow.com/questions/3207219/how-do-i-list-all-files-of-a-directory

# Discussion